# ___Data analysis for the paper draft; with `FRED 4.0`___
--------------------------------------

In [1]:
!python --version

Python 3.14.4


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [89]:
fred = pd.read_csv(r"../../data/chapter2/FRED/FRED4_Entire_Database_2026.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1") # FRED v3
meta = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Column_Definitions_2021.csv", usecols=["column_id", "name", "units"],
                                   index_col="column_id")

lookup = pd.read_csv(r"../../data/chapter2/plant_lookup.csv", low_memory=False, encoding="latin1", usecols=["genus", "apweb.family", # let's stick the the family info from APG website
                                                                                                         "order", "group"], index_col="genus").rename(mapper={"apweb.family": "family"}, axis=1) 

In [109]:
# including all the traits Luke advised

COLLABORATION_GRADIENT_TRAITS = [
    "F00679", # RD
    "F00727", # SRL
    # "F00104", # RCT
]

CONSERVATION_GRADIENT_TRAITS = [
    "F00709", # RTD
    # "F00261", # RN
]

CHOSEN_ROOT_TRAITS = COLLABORATION_GRADIENT_TRAITS + CONSERVATION_GRADIENT_TRAITS

CHOSEN_TRAITS_DICT = {
    "F00709":  "RTD",
    # "F00261":  "RN",
    "F00679":  "RD",
    "F00727":  "SRL",
    # "F00104":  "RCT",
}

PLANT_TAXONOMY_ACCEPTED_COLUMNS = [
    "F01286", # Genus of plant according to The Plant List
    "F01287", # Species epithet of plant according to The Plant List
    "F01289", # Family of plant according to The Plant List
    "F01290"  # Order of plant.
]

BINOMINAL_NAME = ["F01286", "F01287"]
BINOMINAL_NAME_DATA_SOURCE = ["F00018", "F00019"]
ROOT_ORDER = ["F00056"]
CROSSCHECK_COLUMNS = ["F01289", "F01290", "family", "order", "group"]

In [56]:
meta.loc[BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE + CHOSEN_ROOT_TRAITS + ROOT_ORDER, :]

,name,units
column_id,,
F01286,Plant taxonomy_Accepted genus_TPL,NaN
F01287,Plant Taxonomy_Accepted species_TPL,NaN
F00018,Plant taxonomy_Genus_Data Source,NaN
F00019,Plant taxonomy_Species_Data source,NaN
F00679,Root diameter,mm
F00727,Specific root length (SRL),m/g
F00709,Root tissue density (RTD),g/cm3
F00056,Root order,NaN


In [57]:
# records with missing binominal names aren't useful to us 
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].isna().mean() # dropna(subset=BINOMINAL_NAME)

F01286    0.290616
F01287    0.314031
F01289    0.289713
F01290    0.289597
F00679    0.836931
F00727    0.832869
F00709    0.882279
dtype: float64

In [58]:
# that many records with missing binominal names???
fred.loc[:, BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE].isna().sum()

F01286    22539
F01287    24355
F00018    22521
F00019    23283
dtype: int64

In [59]:
# thought we could use the data source's binominal names where FRED's binominal names are missing but looks like that won't help :/
# drop all the rows that do not have genus and species names

fred.dropna(subset=BINOMINAL_NAME, inplace=True)

### ___$1^{st}$ order roots___
__________________

In [60]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Cunninghamia,lanceolata,Cupressaceae,Cupressales,NaN,NaN,NaN,1.0
3,Magnolia,baillonii,Magnoliaceae,Magnoliales,NaN,NaN,NaN,1.0
8,Acacia,auriculiformis,Fabaceae,Fabales,NaN,NaN,NaN,1.0
12,Polyspora,axillaris,Theaceae,Ericales,NaN,NaN,NaN,1.0
15,Acer,negundo,Sapindaceae,Sapindales,NaN,44.500000,0.550000,1.0
...,...,...,...,...,...,...,...,...
70347,Pinus,strobus,Pinaceae,Pinales,0.421385,17.805402,0.337226,1.0
70351,Tsuga,canadensis,Pinaceae,Pinales,0.373780,30.873103,0.204034,1.0
70355,Sciadopitys,verticillata,Sciadopityaceae,Cupressales,0.620386,29.212460,0.125136,1.0
70360,Cephalotaxus,harringtonii,Cephalotaxaceae,Cupressales,0.626291,21.950489,0.125251,1.0


In [61]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
0,Cunninghamia,lanceolata
3,Magnolia,baillonii
8,Acacia,auriculiformis
12,Polyspora,axillaris
15,Acer,negundo
...,...,...
70292,Encephalartos,gratus
70296,Zamia,lucayana
70300,Chamaecyparis,pisifera
70325,Ephedra,distachya


In [62]:
fred.loc[:, CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").isna().mean().rename(index=CHOSEN_TRAITS_DICT)

RD        0.265876
SRL       0.668324
RTD       0.726090
F00056    0.000000
dtype: float64

### ___RD $\le$ 2.0 mm___
-----------------------

In [46]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
51,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN
52,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN
53,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN
54,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN
55,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN
...,...,...,...,...,...,...,...
71899,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091
71900,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155
71901,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379
71902,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331


In [48]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
51,Populus,tremuloides
52,Acer,negundo
53,Juglans,nigra
54,Quercus,rubra
55,Carya,glabra
...,...,...
71810,Sorocea,muriculata
71811,Stachyarrhena,acuminata
71812,Trymatococcus,amazonicus
71813,Zygia,inaequalis


In [49]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").isna().mean().rename(index=CHOSEN_TRAITS_DICT)

F01286    0.000000
F01287    0.000000
F01289    0.000000
F01290    0.000000
RD        0.000000
SRL       0.319051
RTD       0.423580
dtype: float64

### ___RD $\le$ 1.00 mm___
------------------

In [50]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
51,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN
52,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN
53,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN
54,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN
55,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN
...,...,...,...,...,...,...,...
71899,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091
71900,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155
71901,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379
71902,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331


In [51]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
51,Populus,tremuloides
52,Acer,negundo
53,Juglans,nigra
54,Quercus,rubra
55,Carya,glabra
...,...,...
71810,Sorocea,muriculata
71811,Stachyarrhena,acuminata
71812,Trymatococcus,amazonicus
71813,Zygia,inaequalis


In [52]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").isna().mean().rename(index=CHOSEN_TRAITS_DICT)

F01286    0.000000
F01287    0.000000
F01289    0.000000
F01290    0.000000
RD        0.000000
SRL       0.336431
RTD       0.443102
dtype: float64

In [80]:
# final subset for the phylogenetics work;
# using traits - RD, SRL and RTD
# using RD <= 1.00 mm as the criteria

subset = fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00679 <= 1.000").reset_index(drop=True)
subset

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN
1,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN
2,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN
3,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN
4,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
9679,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091,NaN
9680,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155,NaN
9681,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379,NaN
9682,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331,NaN


In [81]:
# examine the root orders within this RD <= 1.00 mm subset
subset.F00056.value_counts(dropna=False)

F00056
NaN    5542
1.0    1908
2.0    1230
3.0     510
4.0     307
5.0     143
6.0      26
7.0       9
8.0       6
9.0       3
Name: count, dtype: int64

In [84]:
# drop records where root order is > 3 ??? Luke advised against mixing the two approaches
subset.query(r"not (F00056 > 3)")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN
1,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN
2,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN
3,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN
4,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
9679,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091,NaN
9680,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155,NaN
9681,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379,NaN
9682,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331,NaN


In [72]:
subset.loc[:, BINOMINAL_NAME].drop_duplicates() # that's a good number of species

,F01286,F01287
0,Populus,tremuloides
1,Acer,negundo
2,Juglans,nigra
3,Quercus,rubra
4,Carya,glabra
...,...,...
9591,Sorocea,muriculata
9592,Stachyarrhena,acuminata
9593,Trymatococcus,amazonicus
9594,Zygia,inaequalis


In [85]:
subset

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN
1,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN
2,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN
3,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN
4,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
9679,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091,NaN
9680,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155,NaN
9681,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379,NaN
9682,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331,NaN


In [92]:
lookup.loc[subset.F01286.unique() ,:]

KeyError: "['Austroblechnum', 'Cranfillia', 'Pectinopitys', 'Helesia', 'Tetrapilus', 'Heptapleurum', 'Veronia', 'Lipschitzia', 'Prasoxylon', 'Buchozia', 'Lysmachia', 'Pseudodictamnus', 'Chrysojasminum', 'Hesperostipa', 'Rubroshorea', 'Cirsim', 'Wurfbainia', 'Andesanthus', 'Imbralyx', 'Alseodaphnopsis', 'Hymenopus', 'Leptobalanus', 'Eumachia'] not in index"

In [107]:
# for the genera that the lookup table has taxonomic data for, do the crosschecking
lookup_merged = pd.merge(left=subset.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].query(r"F01286.isin(@lookup.index)").drop_duplicates(),
                         left_on="F01286", right=lookup, right_index=True, how="left")
lookup_merged

,F01286,F01287,F01289,F01290,family,order,group
0,Populus,tremuloides,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
1,Acer,negundo,Sapindaceae,Sapindales,Sapindaceae,Sapindales,Angiosperms
2,Juglans,nigra,Juglandaceae,Fagales,Juglandaceae,Fagales,Angiosperms
3,Quercus,rubra,Fagaceae,Fagales,Fagaceae,Fagales,Angiosperms
4,Carya,glabra,Juglandaceae,Fagales,Juglandaceae,Fagales,Angiosperms
...,...,...,...,...,...,...,...
9591,Sorocea,muriculata,Moraceae,Rosales,Moraceae,Rosales,Angiosperms
9592,Stachyarrhena,acuminata,Rubiaceae,Gentianales,Rubiaceae,Gentianales,Angiosperms
9593,Trymatococcus,amazonicus,Moraceae,Rosales,Moraceae,Rosales,Angiosperms
9594,Zygia,inaequalis,Fabaceae,Fabales,Fabaceae,Fabales,Angiosperms


In [111]:
lookup_merged.query(r"F01289!=family")

,F01286,F01287,F01289,F01290,family,order,group
499,Hedycarya,arborea,Minimiaceae,Laurales,Monimiaceae,Laurales,Angiosperms
979,Nyssa,sylvatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
1676,Sambucus,williamsii,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
2352,Clinopodium,nepeta,Poaceae,Lamiales,Lamiaceae,Lamiales,Angiosperms
3781,Viburnum,dentatum,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
3898,Sambucus,canadensis,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
4469,Apodytes,dimidiata,Metteniusaceae,Metteniusales,Icacinaceae,Icacinales,Angiosperms
5636,Viburnum,tinus,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
5720,Galearia,maingayi,Fabaceae,Fabales,Pandaceae,Malpighiales,Angiosperms
5883,Nyssa,aquatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms


In [110]:
lookup_merged.query(r"F01289!=family").drop_duplicates(subset=CROSSCHECK_COLUMNS)

,F01286,F01287,F01289,F01290,family,order,group
499,Hedycarya,arborea,Minimiaceae,Laurales,Monimiaceae,Laurales,Angiosperms
979,Nyssa,sylvatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
1676,Sambucus,williamsii,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
2352,Clinopodium,nepeta,Poaceae,Lamiales,Lamiaceae,Lamiales,Angiosperms
4469,Apodytes,dimidiata,Metteniusaceae,Metteniusales,Icacinaceae,Icacinales,Angiosperms
5720,Galearia,maingayi,Fabaceae,Fabales,Pandaceae,Malpighiales,Angiosperms
5930,Moringa,oleifera,Brassicaceae,Brassicales,Moringaceae,Brassicales,Angiosperms
8974,Cephalotaxus,harringtonii,Cephalotaxaceae,Cupressales,Taxaceae,Pinales,Gymnosperms
9239,Cryptocarya,acutifolia,Moraceae,Laurales,Lauraceae,Laurales,Angiosperms
9375,Barringtonia,fusicarpa,Moraceae,Ericales,Lecythidaceae,Ericales,Angiosperms


In [104]:
# examine the genera that are missing in the lookup table
subset.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].query(r"not F01286.isin(@lookup.index)").drop_duplicates().sort_values(by=BINOMINAL_NAME)

,F01286,F01287,F01289,F01290
9264,Alseodaphnopsis,andersonii,Lauraceae,Laurales
8172,Andesanthus,lepidotus,Melastomataceae,Myrtales
483,Austroblechnum,lanceolatum,Blechnaceae,Polypodiales
2106,Buchozia,japonica,Rubiaceae,Gentianales
5589,Chrysojasminum,fruticans,Oleaceae,Lamiales
5766,Cirsim,altissimum,Asteraceae,Asterales
485,Cranfillia,fluviatilis,Blechnaceae,Polypodiales
9493,Eumachia,astrellantha,Rubiaceae,Gentianales
9494,Eumachia,podocephala,Rubiaceae,Gentianales
952,Helesia,tetraptera,Styracaceae,Ericales
